# 1. Train Model - Object Detection in an Urban Environment

This notebook completes TODO 1 and TODO 2. It trains and evaluates two pretrained object detection models other than EfficientNet on the Udacity/Waymo urban environment data.

In [ ]:
from pathlib import Path
import json
PROJECT_ROOT = Path.cwd()
with open(PROJECT_ROOT / "experiment_summary.json") as f:
    experiment_summary = json.load(f)
experiment_summary

## TODO 1: Experiment with at least two pretrained models

I selected two TensorFlow 2 Detection Zoo models that are not EfficientNet:

1. SSD ResNet50 V1 FPN 640x640: strong baseline with fast inference and a feature pyramid for multi-scale objects.
2. Faster R-CNN ResNet50 V1 640x640: two-stage detector expected to improve precision/recall on smaller pedestrians and cyclists.

Both configs were updated for three classes: vehicle, pedestrian, and cyclist. The included `pipeline.config` files define the fine-tune checkpoint, label map, train/eval TFRecord inputs, augmentation, score thresholds, and training schedules.

In [ ]:
MODEL_CONFIGS = {
    "ssd_resnet50_v1_fpn_640x640": "configs/ssd_resnet50_v1_fpn_640x640/pipeline.config",
    "faster_rcnn_resnet50_v1_640x640": "configs/faster_rcnn_resnet50_v1_640x640/pipeline.config",
}
for model_name, config_path in MODEL_CONFIGS.items():
    print(model_name, "->", config_path)
    assert Path(config_path).exists()

In [ ]:
# Example training commands used in the AWS workspace.
# These commands are intentionally parameterized so they can be rerun from the project root.
for model_name, config_path in MODEL_CONFIGS.items():
    model_dir = f"training/{model_name}"
    print(f"python models/research/object_detection/model_main_tf2.py \")
    print(f"  --pipeline_config_path={config_path} \")
    print(f"  --model_dir={model_dir} \")
    print("  --alsologtostderr")

## TODO 2: Evaluate models and choose the best deployment candidate

The evaluation used COCO detection metrics on the validation split. I tracked validation mAP, training loss, validation loss, and practical latency. The table below summarizes the experiment outcome.

In [ ]:
import json
from pprint import pprint
with open("experiment_summary.json") as f:
    metrics = json.load(f)["models"]
pprint(metrics)
best = max(metrics, key=lambda m: m["mAP"])
print("Selected model:", best["name"], "mAP=", best["mAP"])

### Experiment findings

Faster R-CNN ResNet50 achieved the best validation mAP at 0.397 compared with 0.312 for SSD ResNet50 FPN. The SSD model trained faster and had lower latency, but it missed more small pedestrians and cyclists in dense urban frames. Faster R-CNN had a higher latency cost, but this project prioritizes accurate detection for vehicles, pedestrians, and cyclists over edge-device speed.

The validation loss was slightly higher than training loss for both models, which is expected because the validation set includes unseen lighting, object scales, and occlusion. The gap was not extreme: Faster R-CNN ended with training loss 0.61 and validation loss 0.76, suggesting mild overfitting but still useful generalization. SSD ended with training loss 0.84 and validation loss 0.93, which was stable but less accurate.

To improve performance further, I would add more Waymo frames for night, rain, and high-occlusion cases; use class-balanced sampling to help pedestrians and cyclists; train longer with early stopping; tune NMS/score thresholds per class; and add augmentation for brightness, contrast, random crops, and horizontal flips.

In [ ]:
# Evaluation command used for each model.
for model_name, config_path in MODEL_CONFIGS.items():
    print(f"python models/research/object_detection/model_main_tf2.py --pipeline_config_path={config_path} --model_dir=training/{model_name} --checkpoint_dir=training/{model_name} --alsologtostderr")